# Model Prediction & Evaluation
### Breast Cancer Wisconsin Dataset — Small MLP Classifier

**What you'll do in this notebook:**
1. Split data and train a small neural network (MLP)
2. Get prediction *probabilities* and turn them into class predictions
3. Build a confusion matrix and calculate evaluation metrics
4. Visualize ROC & PR curves
5. Deal with class imbalance using weighted metrics

> 💡 Along the way, we ask you to change numbers yourself and watch which effect that change had.


## 0. Setup

Run this once. If you're on a fresh environment, uncomment the install line.

In [ ]:
# !pip install scikit-learn matplotlib numpy --quiet

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
)

RANDOM_STATE = 42  # keep this fixed so everyone gets comparable results
np.random.seed(RANDOM_STATE)

## 1. Load Data & Train/Test Split

The dataset contains 30 numeric features describing cell nuclei, and a binary label:
`0 = malignant`, `1 = benign`.

We split into **train** (model learns from this) and **test** (we evaluate on this — the model never sees it during training). This is what makes our evaluation honest.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

print("Samples:", X.shape[0], "| Features:", X.shape[1])
print("Class counts -> malignant (0):", (y == 0).sum(), " benign (1):", (y == 1).sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,  # keep the same class ratio in both splits
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])

**Note on `stratify=y`:** without it, a random split could accidentally put more malignant cases in one split than the other just by chance. Stratifying keeps the malignant/benign ratio consistent across train and test — important once we talk about class imbalance later.

We also scale the features. MLPs are sensitive to feature scale (unlike e.g. decision trees), so we standardize to mean 0 / std 1. We fit the scaler on the *training* data only, then apply it to both — fitting it on test data would leak information from the test set into training.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 2. Train a Small MLP

Not the focus of this tutorial, but we need a trained model to evaluate. A small multi-layer perceptron (one hidden layer, 16 units) is enough for this dataset.

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(16,),
    max_iter=2000,
    random_state=RANDOM_STATE,
)

mlp.fit(X_train_scaled, y_train)
print("Training done. Iterations used:", mlp.n_iter_)

## 3. From Probabilities to Predictions

The model doesn't output classes directly — it outputs a **probability** for each class. `predict_proba` gives us, for every test sample, the probability of being malignant (column 0) and benign (column 1).

To turn a probability into an actual class prediction, we need a **decision threshold**: if `P(benign) >= threshold`, predict benign, otherwise predict malignant. The standard default is **0.5** — but that's a choice, not a law of nature.

In [ ]:
# Probability of the POSITIVE class. We'll treat "benign" (1) as positive here,
# matching sklearn's default convention for binary labels {0, 1}.
y_proba = mlp.predict_proba(X_test_scaled)[:, 1]  # P(benign)

# --- THE ONE NUMBER YOU'LL CHANGE LATER ---
THRESHOLD = 0.50
# -------------------------------------------

y_pred = (y_proba >= THRESHOLD).astype(int)

print("First 10 probabilities (P(benign)):", np.round(y_proba[:10], 2))
print("First 10 predictions   (0=malignant, 1=benign):", y_pred[:10])

## 4. Confusion Matrix

The confusion matrix counts how predictions line up with reality:

| | Predicted malignant | Predicted benign |
|---|---|---|
| **Actually malignant** | True Negative* | False Positive |
| **Actually benign** | False Negative* | True Positive |

*Naming depends on which class you call "positive" — we're treating **benign = positive**, so:
- **TP** = correctly predicted benign
- **TN** = correctly predicted malignant
- **FP** = predicted benign, but actually malignant ⚠️ *(dangerous: a sick patient sent home)*
- **FN** = predicted malignant, but actually benign *(safe-ish: a healthy patient gets extra tests)*

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix:")
print(cm)
print(f"\nTN={tn}  FP={fp}  FN={fn}  TP={tp}")

ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"]).plot(cmap="Blues")
plt.title(f"Confusion Matrix (threshold = {THRESHOLD})")
plt.show()

## 5. Metrics — Calculated By Hand

All of these come directly from the four numbers above. No magic, just arithmetic:

- **Accuracy** = $\dfrac{TP + TN}{TP+TN+FP+FN}$ — overall, how often is the model right?
- **Precision** = $\dfrac{TP}{TP+FP}$ — of everything predicted *benign*, how much really is?
- **Sensitivity / Recall** = $\dfrac{TP}{TP+FN}$ — of everything *actually* benign, how much did we catch?
  *(Yes — sensitivity and recall are the same thing, just different fields use different names: medicine says "sensitivity", ML papers usually say "recall".)*
- **F1 Score** = $2 \cdot \dfrac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ — harmonic mean of precision & recall, useful when you want one number balancing both.

In [ ]:
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # = recall
f1 = (2 * precision * sensitivity / (precision + sensitivity)
      if (precision + sensitivity) > 0 else 0.0)

print(f"Threshold:          {THRESHOLD}")
print(f"Accuracy:            {accuracy:.3f}")
print(f"Precision:           {precision:.3f}")
print(f"Sensitivity (Recall): {sensitivity:.3f}")
print(f"F1 Score:            {f1:.3f}")

**Sanity check** — scikit-learn has built-in functions for all of these (`accuracy_score`, `precision_score`, `recall_score`, `f1_score`). Feel free to import them and confirm your hand-calculated numbers match. We computed them manually first so the confusion-matrix connection is clear, not just a function call.

## 5b. Try It Yourself: Change the Threshold

Before we go any further — go back to **Section 3**, change `THRESHOLD` to `0.8`, and re-run Sections 3, 4, and 5 (just those cells, top to bottom).

Watch what happens to precision and sensitivity/recall. Then try `0.2` and watch again.

Write down (mentally or in a comment) what you notice:
- Which metric went up, which went down?
- Did they move in the *same* direction, or opposite directions?

Once you've tried both, set the threshold back to `0.5` and continue below — we're about to formalize exactly what you just observed.

## 6. Beyond One Threshold: ROC & PR Curves

You just saw it firsthand: change the threshold, and precision/recall move in opposite directions. Every metric in Section 5 is a snapshot at **one specific threshold**. So what if, instead of manually trying a few values, we computed those metrics at *every* threshold from 0 to 1 at once? Plotting that sweep gives us a curve — this is exactly the trade-off you just felt, now drawn out in full.

### ROC Curve
Plots **True Positive Rate** (= sensitivity/recall) against **False Positive Rate** ($\frac{FP}{FP+TN}$) at every threshold.
**AUC** (Area Under the Curve) compresses the whole curve into one number: the probability that a random benign case gets a higher predicted probability than a random malignant case. `0.5` = random guessing, `1.0` = perfect separation.

### Precision-Recall (PR) Curve
Plots **Precision** against **Recall** at every threshold — the exact two metrics you just watched trade off against each other. More informative than ROC when classes are imbalanced (more on that next section!), because it doesn't involve the (often huge) number of true negatives.

Both curves are **threshold-independent** — they tell you how good the model's probability ranking is, across every cutoff, not just the one you happened to try.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_proba)
ap_score = average_precision_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ROC
axes[0].plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="random guess")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Sensitivity)")
axes[0].set_title("ROC Curve")
axes[0].legend()

# PR
axes[1].plot(recall_curve, precision_curve, label=f"AP = {ap_score:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"ROC AUC:        {auc_score:.3f}")
print(f"Average Precision (PR AUC): {ap_score:.3f}")

**Reading these:** your current threshold corresponds to exactly *one point* on each of these curves. Everything you computed by hand in Section 5 lives somewhere on these lines. Moving the threshold moves you along the curve — it doesn't change the curve itself, because the curve already reflects every possible threshold.

## 7. Class Imbalance

Look back at the class counts from Section 1: malignant and benign aren't perfectly balanced (roughly 37% / 63%). This matters because **standard metrics can be misleading** when classes are imbalanced.

**Why?** Imagine a (bad) model that always predicts "benign", no matter the input. With our class split, it would still get ~63% accuracy — without learning anything. Accuracy alone can hide a model that's failing on the minority class entirely.

**The fix at evaluation time:** weighted metrics. Instead of one global score, compute the metric *per class*, then average — weighting each class either equally (`macro`) or by how many samples it has (`weighted`).

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

print("Class balance in test set:")
print(" malignant:", (y_test == 0).sum(), f"({(y_test==0).mean():.1%})")
print(" benign:   ", (y_test == 1).sum(), f"({(y_test==1).mean():.1%})")
print()

# 'macro'    -> average metric per class, unweighted (treats classes as equally important)
# 'weighted' -> average metric per class, weighted by class size (accounts for imbalance)
precision_macro = precision_score(y_test, y_pred, average="macro")
recall_macro = recall_score(y_test, y_pred, average="macro")
f1_macro = f1_score(y_test, y_pred, average="macro")

precision_weighted = precision_score(y_test, y_pred, average="weighted")
recall_weighted = recall_score(y_test, y_pred, average="weighted")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"{'Metric':<12}{'Macro avg':>12}{'Weighted avg':>15}")
print(f"{'Precision':<12}{precision_macro:>12.3f}{precision_weighted:>15.3f}")
print(f"{'Recall':<12}{recall_macro:>12.3f}{recall_weighted:>15.3f}")
print(f"{'F1':<12}{f1_macro:>12.3f}{f1_weighted:>15.3f}")

print("\nFull per-class report:")
print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

**One more lever (optional, just for awareness):** everything above fixes the imbalance problem *at evaluation time* — after the model is already trained. You can also address it *at training time* by passing `class_weight="balanced"` to many sklearn models, which tells the model to pay more attention to the minority class while learning. Different tool, same underlying problem.

## 8. Your Turn 🔁 (Full Replay)

Earlier in Section 5b you already saw precision and recall trade off at a couple of threshold values. Now let's run the *whole* pipeline again at a new threshold, including the curves.

Go back to **Section 3**, change `THRESHOLD` to `0.3`, and re-run every cell from Section 3 onward (`Cell` → `Run All Below`, or just re-run them one by one).

Watch what happens to:
- The confusion matrix (Section 4) and the hand-computed metrics (Section 5)
- The ROC and PR plots (Section 6) — specifically, **where your new threshold's point would sit on each curve**
- **What stays exactly the same:** the curves themselves, and the AUC/AP scores — only *where you'd sit on the curve* changes, because the curve already accounts for every possible threshold.

Try `0.8` too, and answer for yourself:
- What does lowering the threshold do to false negatives? Why might that matter more in a medical context than false positives — or does it?
- At what threshold would you personally be comfortable deploying this model, and why?
